# NYC TLC Trip Data — Exploratory Analysis

Exploratory analysis across all four NYC taxi/FHV datasets using DuckDB to query parquet files directly.
No data is loaded into memory — DuckDB reads only the columns and row groups needed for each query.

In [ ]:
import sys
import plotly.graph_objects as go

from utils.eda_data_loading import get_connection, summarize_available_data

con = get_connection()

# Cap DuckDB memory to leave headroom for OS/JupyterLab
con.sql("SET memory_limit = '50GB'")

print("Connection ready. Available data:")
print()
summarize_available_data(con)

Connection ready. Available data:

yellow:  1,822,655,636 rows (2009-01 to 2025-11)
green:     84,027,827 rows (2014-01 to 2025-11)
  fhv:    775,053,951 rows (2015-01 to 2025-10)
fhvhv:  1,439,941,036 rows (2019-02 to 2025-11)


In [6]:
def plot_histogram(con, query, xlabel, title, color="steelblue"):
    """Plot a pre-aggregated histogram from a DuckDB query.
    
    The query must return two columns: bin_start and count.
    """
    result = con.sql(query).fetchnumpy()
    bins = result["bin_start"]
    counts = result["count"]
    
    if len(bins) == 0:
        print(f"No data for: {title}")
        return
    
    width = bins[1] - bins[0] if len(bins) > 1 else 1
    
    fig = go.Figure(go.Bar(
        x=bins, y=counts,
        width=width * 0.9,
        marker_color=color,
    ))
    fig.update_layout(
        title=title,
        xaxis_title=xlabel,
        yaxis_title="Trip count",
        height=400,
    )
    fig.show()

---
## 1. Yellow Taxi

### 1.1 Row counts by year

In [7]:
con.sql("""
    SELECT
        year(CAST(trip_pickup_datetime AS TIMESTAMP)) AS year,
        count(*) AS trips,
    FROM yellow_trips
    GROUP BY year
    ORDER BY year
""").show(max_rows=50)

┌───────┬───────────┐
│ year  │   trips   │
│ int64 │   int64   │
├───────┼───────────┤
│  2001 │        27 │
│  2002 │       498 │
│  2003 │        50 │
│  2004 │         1 │
│  2007 │         1 │
│  2008 │       771 │
│  2009 │ 170897359 │
│  2010 │ 169001154 │
│  2011 │ 176887272 │
│  2012 │ 171359008 │
│  2013 │ 171816340 │
│  2014 │ 165447580 │
│  2015 │ 146039232 │
│  2016 │ 131131805 │
│  2017 │ 113500386 │
│  2018 │ 102870524 │
│  2019 │  84597309 │
│  2020 │  24649266 │
│  2021 │  30903983 │
│  2022 │  39655622 │
│  2023 │  38310138 │
│  2024 │  41169691 │
│  2025 │  44417572 │
│  2026 │         5 │
│  2028 │         1 │
│  2029 │         7 │
│  2031 │         2 │
│  2032 │         1 │
│  2033 │         3 │
│  2037 │         1 │
│  2038 │         4 │
│  2041 │         3 │
│  2042 │         1 │
│  2053 │         2 │
│  2058 │         3 │
│  2066 │         1 │
│  2070 │         1 │
│  2084 │         8 │
│  2088 │         2 │
│  2090 │         1 │
│  2098 │         1 │
├───────┴─

### 1.2 Descriptive statistics — numeric columns

In [8]:
con.sql("""
    SELECT
        count(*) AS n,
        avg(passenger_count) AS avg_passengers,
        avg(trip_distance) AS avg_distance,
        avg(fare_amt) AS avg_fare,
        avg(tip_amt) AS avg_tip,
        avg(tolls_amt) AS avg_tolls,
        avg(total_amt) AS avg_total,
        avg(surcharge) AS avg_surcharge
    FROM yellow_trips
""").show()

┌────────────┬────────────────────┬───────────────────┬────────────────────┬───────────────────┬─────────────────────┬────────────────────┬─────────────────────┐
│     n      │   avg_passengers   │   avg_distance    │      avg_fare      │      avg_tip      │      avg_tolls      │     avg_total      │    avg_surcharge    │
│   int64    │       double       │      double       │       double       │      double       │       double        │       double       │       double        │
├────────────┼────────────────────┼───────────────────┼────────────────────┼───────────────────┼─────────────────────┼────────────────────┼─────────────────────┤
│ 1822655636 │ 1.6324299454639852 │ 5.767829271784344 │ 12.247597957772987 │ 1.622243763472039 │ 0.26068418048876046 │ 15.518569212149893 │ 0.45557850310240194 │
└────────────┴────────────────────┴───────────────────┴────────────────────┴───────────────────┴─────────────────────┴────────────────────┴─────────────────────┘



In [9]:
con.sql("""
    SELECT
        'trip_distance' AS metric,
        min(trip_distance) AS min,
        approx_quantile(trip_distance, 0.25) AS p25,
        approx_quantile(trip_distance, 0.50) AS median,
        approx_quantile(trip_distance, 0.75) AS p75,
        max(trip_distance) AS max,
        stddev(trip_distance) AS stddev
    FROM yellow_trips
    UNION ALL
    SELECT
        'fare_amt',
        min(fare_amt), approx_quantile(fare_amt, 0.25), approx_quantile(fare_amt, 0.50),
        approx_quantile(fare_amt, 0.75), max(fare_amt), stddev(fare_amt)
    FROM yellow_trips
    UNION ALL
    SELECT
        'tip_amt',
        min(tip_amt), approx_quantile(tip_amt, 0.25), approx_quantile(tip_amt, 0.50),
        approx_quantile(tip_amt, 0.75), max(tip_amt), stddev(tip_amt)
    FROM yellow_trips
    UNION ALL
    SELECT
        'total_amt',
        min(total_amt), approx_quantile(total_amt, 0.25), approx_quantile(total_amt, 0.50),
        approx_quantile(total_amt, 0.75), max(total_amt), stddev(total_amt)
    FROM yellow_trips
""").show()

┌───────────────┬──────────────┬───────────────────┬────────────────────┬────────────────────┬──────────────┬────────────────────┐
│    metric     │     min      │        p25        │       median       │        p75         │     max      │       stddev       │
│    varchar    │    double    │      double       │       double       │       double       │    double    │       double       │
├───────────────┼──────────────┼───────────────────┼────────────────────┼────────────────────┼──────────────┼────────────────────┤
│ trip_distance │  -40840124.4 │ 1.012536206049514 │   1.73054426426058 │ 3.1704278588121317 │  134619063.1 │ 5933.0194514779705 │
│ fare_amt      │ -133391414.0 │ 6.300043011779454 │  9.001977244607566 │ 13.892591423349687 │    998310.03 │ 3333.8871664205644 │
│ tip_amt       │   -1677720.1 │               0.0 │ 0.9999987992719612 │ 2.0943291159749533 │ 133391363.53 │  3295.199274503377 │
│ total_amt     │  -21474830.0 │  7.97966544648807 │  11.42130397277474 │  17.41565

### 1.3 Histograms

In [10]:
plot_histogram(con, """
    SELECT floor(trip_distance / 0.5) * 0.5 AS bin_start, count(*) AS count
    FROM yellow_trips
    WHERE trip_distance BETWEEN 0 AND 30
    GROUP BY bin_start ORDER BY bin_start
""", "Miles", "Yellow Taxi — Trip Distance Distribution")

In [11]:
plot_histogram(con, """
    SELECT floor(fare_amt / 2) * 2 AS bin_start, count(*) AS count
    FROM yellow_trips
    WHERE fare_amt BETWEEN 0 AND 100
    GROUP BY bin_start ORDER BY bin_start
""", "Fare ($)", "Yellow Taxi — Fare Distribution")

In [12]:
plot_histogram(con, """
    SELECT floor(tip_amt / 1) * 1 AS bin_start, count(*) AS count
    FROM yellow_trips
    WHERE tip_amt BETWEEN 0 AND 30
    GROUP BY bin_start ORDER BY bin_start
""", "Tip ($)", "Yellow Taxi — Tip Distribution")

In [13]:
plot_histogram(con, """
    SELECT passenger_count AS bin_start, count(*) AS count
    FROM yellow_trips
    WHERE passenger_count BETWEEN 1 AND 9
    GROUP BY bin_start ORDER BY bin_start
""", "Passengers", "Yellow Taxi — Passenger Count Distribution")

In [14]:
# Trip duration in minutes (derived from timestamps)
plot_histogram(con, """
    SELECT floor(duration_min / 2) * 2 AS bin_start, count(*) AS count
    FROM (
        SELECT
            date_diff('minute',
                CAST(trip_pickup_datetime AS TIMESTAMP),
                CAST(trip_dropoff_datetime AS TIMESTAMP)
            ) AS duration_min
        FROM yellow_trips
    )
    WHERE duration_min BETWEEN 1 AND 60
    GROUP BY bin_start ORDER BY bin_start
""", "Minutes", "Yellow Taxi — Trip Duration Distribution")

---
## 2. Green Taxi

### 2.1 Row counts by year

In [15]:
con.sql("""
    SELECT
        year(CAST(lpep_pickup_datetime AS TIMESTAMP)) AS year,
        count(*) AS trips
    FROM green_trips
    GROUP BY year
    ORDER BY year
""").show(max_rows=50)

┌───────┬──────────┐
│ year  │  trips   │
│ int64 │  int64   │
├───────┼──────────┤
│  2008 │      114 │
│  2009 │      315 │
│  2010 │      348 │
│  2012 │        3 │
│  2014 │ 15837009 │
│  2015 │ 19233765 │
│  2016 │ 16385541 │
│  2017 │ 11736906 │
│  2018 │  8899314 │
│  2019 │  6300814 │
│  2020 │  1734166 │
│  2021 │  1068729 │
│  2022 │   840394 │
│  2023 │   787055 │
│  2024 │   660204 │
│  2025 │   543144 │
│  2030 │        2 │
│  2035 │        1 │
│  2041 │        1 │
│  2062 │        1 │
│  2081 │        1 │
├───────┴──────────┤
│     21 rows      │
└──────────────────┘



### 2.2 Descriptive statistics

In [16]:
con.sql("""
    SELECT
        count(*) AS n,
        avg(passenger_count) AS avg_passengers,
        avg(trip_distance) AS avg_distance,
        avg(fare_amount) AS avg_fare,
        avg(tip_amount) AS avg_tip,
        avg(tolls_amount) AS avg_tolls,
        avg(total_amount) AS avg_total,
        avg(extra) AS avg_extra
    FROM green_trips
""").show()

┌──────────┬────────────────────┬──────────────────┬────────────────────┬────────────────────┬─────────────────────┬───────────────────┬─────────────────────┐
│    n     │   avg_passengers   │   avg_distance   │      avg_fare      │      avg_tip       │      avg_tolls      │     avg_total     │      avg_extra      │
│  int64   │       double       │      double      │       double       │       double       │       double        │      double       │       double        │
├──────────┼────────────────────┼──────────────────┼────────────────────┼────────────────────┼─────────────────────┼───────────────────┼─────────────────────┤
│ 84027827 │ 1.3686633971688984 │ 6.26061060914939 │ 12.995357783798648 │ 1.1831210802344736 │ 0.14886752908654838 │ 15.55104859736537 │ 0.40084423020959653 │
└──────────┴────────────────────┴──────────────────┴────────────────────┴────────────────────┴─────────────────────┴───────────────────┴─────────────────────┘



In [17]:
con.sql("""
    SELECT
        'trip_distance' AS metric,
        min(trip_distance) AS min,
        approx_quantile(trip_distance, 0.25) AS p25,
        approx_quantile(trip_distance, 0.50) AS median,
        approx_quantile(trip_distance, 0.75) AS p75,
        max(trip_distance) AS max,
        stddev(trip_distance) AS stddev
    FROM green_trips
    UNION ALL
    SELECT
        'fare_amount',
        min(fare_amount), approx_quantile(fare_amount, 0.25), approx_quantile(fare_amount, 0.50),
        approx_quantile(fare_amount, 0.75), max(fare_amount), stddev(fare_amount)
    FROM green_trips
    UNION ALL
    SELECT
        'tip_amount',
        min(tip_amount), approx_quantile(tip_amount, 0.25), approx_quantile(tip_amount, 0.50),
        approx_quantile(tip_amount, 0.75), max(tip_amount), stddev(tip_amount)
    FROM green_trips
    UNION ALL
    SELECT
        'total_amount',
        min(total_amount), approx_quantile(total_amount, 0.25), approx_quantile(total_amount, 0.50),
        approx_quantile(total_amount, 0.75), max(total_amount), stddev(total_amount)
    FROM green_trips
""").show()

┌───────────────┬───────────┬────────────────────┬────────────────────┬────────────────────┬───────────┬────────────────────┐
│    metric     │    min    │        p25         │       median       │        p75         │    max    │       stddev       │
│    varchar    │  double   │       double       │       double       │       double       │  double   │       double       │
├───────────────┼───────────┼────────────────────┼────────────────────┼────────────────────┼───────────┼────────────────────┤
│ trip_distance │ -20329.08 │ 1.0620977941649359 │ 1.9149472034265262 │ 3.7025470511175294 │ 360068.14 │  614.7758636209679 │
│ fare_amount   │    -890.0 │  6.500012868691702 │   9.70819077998975 │ 15.855728171099635 │  10445.84 │  11.54336067705893 │
│ tip_amount    │    -101.0 │                0.0 │                0.0 │ 1.9558287997127373 │   2017.73 │  2.660764264027039 │
│ total_amount  │    -890.3 │  8.116150036945427 │ 11.762267383422339 │ 18.779089585500913 │ 989970.39 │ 109.360462764

### 2.3 Histograms

In [18]:
plot_histogram(con, """
    SELECT floor(trip_distance / 0.5) * 0.5 AS bin_start, count(*) AS count
    FROM green_trips
    WHERE trip_distance BETWEEN 0 AND 30
    GROUP BY bin_start ORDER BY bin_start
""", "Miles", "Green Taxi — Trip Distance Distribution", color="seagreen")

In [19]:
plot_histogram(con, """
    SELECT floor(fare_amount / 2) * 2 AS bin_start, count(*) AS count
    FROM green_trips
    WHERE fare_amount BETWEEN 0 AND 100
    GROUP BY bin_start ORDER BY bin_start
""", "Fare ($)", "Green Taxi — Fare Distribution", color="seagreen")

In [20]:
plot_histogram(con, """
    SELECT floor(tip_amount / 1) * 1 AS bin_start, count(*) AS count
    FROM green_trips
    WHERE tip_amount BETWEEN 0 AND 30
    GROUP BY bin_start ORDER BY bin_start
""", "Tip ($)", "Green Taxi — Tip Distribution", color="seagreen")

In [21]:
plot_histogram(con, """
    SELECT passenger_count AS bin_start, count(*) AS count
    FROM green_trips
    WHERE passenger_count BETWEEN 1 AND 9
    GROUP BY bin_start ORDER BY bin_start
""", "Passengers", "Green Taxi — Passenger Count Distribution", color="seagreen")

In [22]:
plot_histogram(con, """
    SELECT floor(duration_min / 2) * 2 AS bin_start, count(*) AS count
    FROM (
        SELECT
            date_diff('minute',
                CAST(lpep_pickup_datetime AS TIMESTAMP),
                CAST(lpep_dropoff_datetime AS TIMESTAMP)
            ) AS duration_min
        FROM green_trips
    )
    WHERE duration_min BETWEEN 1 AND 60
    GROUP BY bin_start ORDER BY bin_start
""", "Minutes", "Green Taxi — Trip Duration Distribution", color="seagreen")

---
## 3. FHV (For-Hire Vehicles)

### 3.1 Row counts by year

In [23]:
con.sql("""
    SELECT
        year(CAST(pickup_datetime AS TIMESTAMP)) AS year,
        count(*) AS trips
    FROM fhv_trips
    GROUP BY year
    ORDER BY year
""").show(max_rows=50)

┌───────┬───────────┐
│ year  │   trips   │
│ int64 │   int64   │
├───────┼───────────┤
│  2015 │  63388532 │
│  2016 │ 132114083 │
│  2017 │ 192309557 │
│  2018 │ 260874754 │
│  2019 │  43261276 │
│  2020 │  14945465 │
│  2021 │  14805265 │
│  2022 │  14511664 │
│  2023 │  15858639 │
│  2024 │   9501967 │
│  2025 │  13482749 │
├───────┴───────────┤
│      11 rows      │
└───────────────────┘



### 3.2 Descriptive statistics

FHV data is sparse — only 7 columns, mostly categorical. The main numeric fields are the location IDs.

In [24]:
con.sql("""
    SELECT
        count(*) AS total_trips,
        count(DISTINCT dispatching_base_num) AS unique_bases,
        count(DISTINCT CAST(pulocationid AS INTEGER)) AS unique_pickup_zones,
        count(DISTINCT CAST(dolocationid AS INTEGER)) AS unique_dropoff_zones,
        count(sr_flag) AS shared_rides
    FROM fhv_trips
""").show()

┌─────────────┬──────────────┬─────────────────────┬──────────────────────┬──────────────┐
│ total_trips │ unique_bases │ unique_pickup_zones │ unique_dropoff_zones │ shared_rides │
│    int64    │    int64     │        int64        │        int64         │    int64     │
├─────────────┼──────────────┼─────────────────────┼──────────────────────┼──────────────┤
│   775053951 │         1421 │                 266 │                  266 │     88943351 │
└─────────────┴──────────────┴─────────────────────┴──────────────────────┴──────────────┘



In [25]:
# Trip duration is the main continuous variable we can derive
con.sql("""
    SELECT
        'duration_min' AS metric,
        min(duration_min) AS min,
        approx_quantile(duration_min, 0.25) AS p25,
        approx_quantile(duration_min, 0.50) AS median,
        approx_quantile(duration_min, 0.75) AS p75,
        max(duration_min) AS max,
        stddev(duration_min) AS stddev
    FROM (
        SELECT date_diff('minute',
            CAST(pickup_datetime AS TIMESTAMP),
            CAST(dropoff_datetime AS TIMESTAMP)
        ) AS duration_min
        FROM fhv_trips
        WHERE dropoff_datetime IS NOT NULL
    )
""").show()

┌──────────────┬───────────┬───────────┬────────┬───────┬────────────┬───────────────────┐
│    metric    │    min    │    p25    │ median │  p75  │    max     │      stddev       │
│   varchar    │   int64   │   int64   │ int64  │ int64 │   int64    │      double       │
├──────────────┼───────────┼───────────┼────────┼───────┼────────────┼───────────────────┤
│ duration_min │ -55516277 │ -14268176 │     10 │    21 │ 1673570885 │ 6885082.826745633 │
└──────────────┴───────────┴───────────┴────────┴───────┴────────────┴───────────────────┘



### 3.3 Histograms

In [26]:
plot_histogram(con, """
    SELECT floor(duration_min / 2) * 2 AS bin_start, count(*) AS count
    FROM (
        SELECT date_diff('minute',
            CAST(pickup_datetime AS TIMESTAMP),
            CAST(dropoff_datetime AS TIMESTAMP)
        ) AS duration_min
        FROM fhv_trips
        WHERE dropoff_datetime IS NOT NULL
    )
    WHERE duration_min BETWEEN 1 AND 60
    GROUP BY bin_start ORDER BY bin_start
""", "Minutes", "FHV — Trip Duration Distribution", color="coral")

In [27]:
# Top 20 pickup locations
con.sql("""
    SELECT
        CAST(pulocationid AS INTEGER) AS pickup_zone,
        count(*) AS trips
    FROM fhv_trips
    WHERE pulocationid IS NOT NULL
    GROUP BY pickup_zone
    ORDER BY trips DESC
    LIMIT 20
""").show()

┌─────────────┬──────────┐
│ pickup_zone │  trips   │
│    int32    │  int64   │
├─────────────┼──────────┤
│         264 │ 26745115 │
│          79 │ 10445424 │
│         161 │  9768254 │
│         234 │  9095959 │
│         132 │  8820226 │
│          48 │  8303381 │
│         231 │  8294719 │
│         138 │  8189570 │
│          61 │  7521743 │
│         230 │  7350072 │
│         170 │  7258625 │
│          68 │  7219087 │
│         148 │  7205171 │
│         164 │  7021560 │
│         246 │  6749820 │
│         162 │  6582355 │
│         249 │  6560608 │
│         107 │  6462739 │
│         255 │  6356254 │
│         237 │  6304087 │
├─────────────┴──────────┤
│ 20 rows      2 columns │
└────────────────────────┘



---
## 4. FHVHV (High-Volume For-Hire Vehicles — Uber/Lyft)

### 4.1 Row counts by year

In [28]:
con.sql("""
    SELECT
        year(CAST(pickup_datetime AS TIMESTAMP)) AS year,
        count(*) AS trips
    FROM fhvhv_trips
    GROUP BY year
    ORDER BY year
""").show(max_rows=50)

┌───────┬───────────┐
│ year  │   trips   │
│ int64 │   int64   │
├───────┼───────────┤
│  2019 │ 234630264 │
│  2020 │ 143309871 │
│  2021 │ 174596652 │
│  2022 │ 193962535 │
│  2023 │ 232490020 │
│  2024 │ 239470448 │
│  2025 │ 221481246 │
└───────┴───────────┘



### 4.2 Descriptive statistics

In [29]:
con.sql("""
    SELECT
        count(*) AS n,
        avg(trip_miles) AS avg_miles,
        avg(trip_time) / 60.0 AS avg_duration_min,
        avg(base_passenger_fare) AS avg_fare,
        avg(tips) AS avg_tips,
        avg(driver_pay) AS avg_driver_pay,
        avg(tolls) AS avg_tolls,
        avg(congestion_surcharge) AS avg_congestion
    FROM fhvhv_trips
""").show()

┌────────────┬───────────────────┬───────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬───────────────────┐
│     n      │     avg_miles     │ avg_duration_min  │      avg_fare      │      avg_tips      │  avg_driver_pay   │     avg_tolls      │  avg_congestion   │
│   int64    │      double       │      double       │       double       │       double       │      double       │       double       │      double       │
├────────────┼───────────────────┼───────────────────┼────────────────────┼────────────────────┼───────────────────┼────────────────────┼───────────────────┤
│ 1439941036 │ 4.934831971377857 │ 19.26862918864686 │ 22.885480926957644 │ 0.9589677017862643 │ 18.13091146803926 │ 1.0278883509916665 │ 1.019982174238594 │
└────────────┴───────────────────┴───────────────────┴────────────────────┴────────────────────┴───────────────────┴────────────────────┴───────────────────┘



In [30]:
con.sql("""
    SELECT
        'trip_miles' AS metric,
        min(trip_miles) AS min,
        approx_quantile(trip_miles, 0.25) AS p25,
        approx_quantile(trip_miles, 0.50) AS median,
        approx_quantile(trip_miles, 0.75) AS p75,
        max(trip_miles) AS max,
        stddev(trip_miles) AS stddev
    FROM fhvhv_trips
    UNION ALL
    SELECT
        'trip_time_min',
        min(trip_time)/60.0, approx_quantile(trip_time, 0.25)/60.0,
        approx_quantile(trip_time, 0.50)/60.0, approx_quantile(trip_time, 0.75)/60.0,
        max(trip_time)/60.0, stddev(trip_time)/60.0
    FROM fhvhv_trips
    UNION ALL
    SELECT
        'base_passenger_fare',
        min(base_passenger_fare), approx_quantile(base_passenger_fare, 0.25),
        approx_quantile(base_passenger_fare, 0.50), approx_quantile(base_passenger_fare, 0.75),
        max(base_passenger_fare), stddev(base_passenger_fare)
    FROM fhvhv_trips
    UNION ALL
    SELECT
        'driver_pay',
        min(driver_pay), approx_quantile(driver_pay, 0.25), approx_quantile(driver_pay, 0.50),
        approx_quantile(driver_pay, 0.75), max(driver_pay), stddev(driver_pay)
    FROM fhvhv_trips
    UNION ALL
    SELECT
        'tips',
        min(tips), approx_quantile(tips, 0.25), approx_quantile(tips, 0.50),
        approx_quantile(tips, 0.75), max(tips), stddev(tips)
    FROM fhvhv_trips
""").show()

┌─────────────────────┬──────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┐
│       metric        │   min    │        p25         │       median       │        p75         │        max        │       stddev       │
│       varchar       │  double  │       double       │       double       │       double       │      double       │       double       │
├─────────────────────┼──────────┼────────────────────┼────────────────────┼────────────────────┼───────────────────┼────────────────────┤
│ trip_miles          │      0.0 │ 1.5834218795938257 │ 2.9770158813996783 │  6.111568279621799 │           5380.78 │  5.727126579771153 │
│ trip_time_min       │      0.0 │                9.8 │ 15.683333333333334 │ 24.666666666666668 │ 4012.733333333333 │ 13.840979868819291 │
│ base_passenger_fare │ -1969.59 │ 10.552422742899978 │  16.95727348116082 │ 27.881503759535615 │           8157.74 │  20.64315643892698 │
│ driver_pay          │ -68

### 4.3 Histograms

In [31]:
plot_histogram(con, """
    SELECT floor(trip_miles / 0.5) * 0.5 AS bin_start, count(*) AS count
    FROM fhvhv_trips
    WHERE trip_miles BETWEEN 0 AND 30
    GROUP BY bin_start ORDER BY bin_start
""", "Miles", "FHVHV — Trip Distance Distribution", color="mediumpurple")

In [32]:
plot_histogram(con, """
    SELECT floor(trip_time / 120) * 2 AS bin_start, count(*) AS count
    FROM fhvhv_trips
    WHERE trip_time BETWEEN 60 AND 3600
    GROUP BY bin_start ORDER BY bin_start
""", "Minutes", "FHVHV — Trip Duration Distribution", color="mediumpurple")

In [33]:
plot_histogram(con, """
    SELECT floor(base_passenger_fare / 2) * 2 AS bin_start, count(*) AS count
    FROM fhvhv_trips
    WHERE base_passenger_fare BETWEEN 0 AND 100
    GROUP BY bin_start ORDER BY bin_start
""", "Fare ($)", "FHVHV — Base Passenger Fare Distribution", color="mediumpurple")

In [34]:
plot_histogram(con, """
    SELECT floor(driver_pay / 2) * 2 AS bin_start, count(*) AS count
    FROM fhvhv_trips
    WHERE driver_pay BETWEEN 0 AND 80
    GROUP BY bin_start ORDER BY bin_start
""", "Driver Pay ($)", "FHVHV — Driver Pay Distribution", color="mediumpurple")

In [35]:
plot_histogram(con, """
    SELECT floor(tips / 1) * 1 AS bin_start, count(*) AS count
    FROM fhvhv_trips
    WHERE tips BETWEEN 0 AND 30
    GROUP BY bin_start ORDER BY bin_start
""", "Tips ($)", "FHVHV — Tips Distribution", color="mediumpurple")

### 4.4 Uber vs Lyft breakdown

In [36]:
con.sql("""
    SELECT
        CASE hvfhs_license_num
            WHEN 'HV0003' THEN 'Uber'
            WHEN 'HV0005' THEN 'Lyft'
            ELSE hvfhs_license_num
        END AS service,
        count(*) AS trips,
        avg(trip_miles) AS avg_miles,
        avg(trip_time) / 60.0 AS avg_duration_min,
        avg(base_passenger_fare) AS avg_fare,
        avg(tips) AS avg_tips,
        avg(driver_pay) AS avg_driver_pay
    FROM fhvhv_trips
    GROUP BY service
    ORDER BY trips DESC
""").show()

┌─────────┬────────────┬────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┐
│ service │   trips    │     avg_miles      │  avg_duration_min  │      avg_fare      │      avg_tips       │   avg_driver_pay   │
│ varchar │   int64    │       double       │       double       │       double       │       double        │       double       │
├─────────┼────────────┼────────────────────┼────────────────────┼────────────────────┼─────────────────────┼────────────────────┤
│ Uber    │ 1041138750 │  4.933196459050238 │ 19.165176630012088 │ 23.177143876853027 │  0.9275875762064931 │ 18.685090940284045 │
│ Lyft    │  378528395 │   4.99022739985485 │   19.5363914299833 │  22.56136748075536 │  1.0829593619786682 │ 17.241877207941354 │
│ HV0004  │   13884957 │ 3.8640757490282303 │ 20.739949974878087 │  12.46155694397621 │ 0.21723674909471988 │ 2.1803995165415366 │
│ HV0002  │    6388934 │  4.246369190854094 │ 17.065386875703105 │  17.213216963882